In [1]:
%pip install biopython

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


collecting the hemoglobin gene from 7 animals, putting them side by side, and seeing how similar they are.

In [14]:
from Bio import Entrez, SeqIO, Align
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import time



In [26]:
Entrez.email = "your@email.com" # enter your eamil

GENE = "HBB"     # Hemoglobin subunit beta( provids instruction for making beta-globin)
SPECIES = [
    "Homo sapiens",
    "Pan troglodytes",          # chimpanzee
    "Mus musculus",             # mouse
    "Rattus norvegicus",        # rat
    "Canis lupus familiaris",   # dog
    "Bos taurus",               # cow
    "Equus caballus",           # horse
]


def find_record_id(gene, species):
    query = f'{gene}[Gene Name] AND "{species}"[Organism] AND refseq[filter] AND mRNA[Filter]'
    result  = Entrez.read(Entrez.esearch(db = "nucleotide", term = query, retmax = 1))  #Entrez.esearch(...) sends your question to NCBI. NCBI replies with raw XML text (messy, hard to use).
    return result["IdList"][0] if result["IdList"] else None   #Entrez.read(...) converts that XML into normal Python data you can work with.
     

# hi = find_record_id(GENE,"Homo sapiens")



def get_cds_and_protein(uid):
    handle = Entrez.efetch(db = "nucleotide", id = uid, rettype = "gb", retmode = "text")
    rec = SeqIO.read(handle,"genbank")
    for feature in rec.features:
        if feature.type == "CDS":
            cds = feature.extract(rec.seq)
            protein = feature.qualifiers["translation"][0]
            return rec, cds, protein
    return rec, None, None

cds_records, protein_records = [], []

MANUAL = {
       "Mus musculus": "NM_008220",
       "Canis lupus familiaris": "NM_001270883.1",
   }

for species in SPECIES:
    uid = MANUAL.get(species) or find_record_id(GENE, species)
    if uid is None:
        print(f"[SKIP] {species}: nothing found")
        continue

    rec, cds, protein = get_cds_and_protein(uid)
    if cds is None:
        print(f"[SKIP] {species}: no CDS annotation in {rec.id}")
        continue

    label = species.replace(" ", "_")
    print(f"{species:25} {rec.id:14} CDS={len(cds)} bp  protein={len(protein)} aa")
    print(f"   -> {rec.description[:80]}")

    cds_records.append(SeqRecord(cds, id=label, description=rec.id))
    protein_records.append(SeqRecord(Seq(protein), id=label, description=rec.id))
    time.sleep(0.5)

SeqIO.write(cds_records, f"{GENE}_cds.fasta", "fasta")
SeqIO.write(protein_records, f"{GENE}_protein.fasta", "fasta")
print(f"\nSaved {len(protein_records)} species")

Homo sapiens              NM_000518.5    CDS=444 bp  protein=147 aa
   -> Homo sapiens hemoglobin subunit beta (HBB), mRNA
Pan troglodytes           XM_508242.5    CDS=444 bp  protein=147 aa
   -> PREDICTED: Pan troglodytes hemoglobin subunit beta (HBB), mRNA
Mus musculus              NM_008220.5    CDS=444 bp  protein=147 aa
   -> Mus musculus hemoglobin, beta adult t chain (Hbb-bt), mRNA
Rattus norvegicus         NM_033234.1    CDS=444 bp  protein=147 aa
   -> Rattus norvegicus hemoglobin, beta adult major chain (Hbb-b1), mRNA
Canis lupus familiaris    NM_001270883.1 CDS=444 bp  protein=147 aa
   -> Canis lupus familiaris hemoglobin subunit beta-like (LOC609402), mRNA
Bos taurus                NM_173917.2    CDS=438 bp  protein=145 aa
   -> Bos taurus hemoglobin, beta (HBB), mRNA
Equus caballus            NM_001164018.1 CDS=444 bp  protein=147 aa
   -> Equus caballus hemoglobin, beta (HBB), mRNA

Saved 7 species


6 species have 147 aa, and cow has 145 aa.
Mouse is Hbb-bt, rat is Hbb-b1, dog is a LOC gene, and chimp is XM_ (predicted).

In [27]:
aligner = Align.PairwiseAligner()
aligner.mode = "global"

human_protein = str(protein_records[0].seq)   # Homo sapiens was first

def identity(a, b):
    aln = aligner.align(a, b)[0]
    m = sum(x == y for x, y in zip(aln[0], aln[1]) if x != "-" and y != "-")
    return round(100 * m / len(aln[0]), 1)

for acc in ["NM_001270883.1", "NM_001270884.1"]:
    uid = Entrez.read(Entrez.esearch(db="nucleotide", term=acc))["IdList"][0]
    rec, cds, protein = get_cds_and_protein(uid)
    print(acc, "|", len(protein), "aa | identity to human HBB:", identity(human_protein, protein), "%")

NM_001270883.1 | 147 aa | identity to human HBB: 89.8 %
NM_001270884.1 | 147 aa | identity to human HBB: 89.8 %


Test the dog candidates against human HBB

Add this to your notebook (it reuses get_cds_and_protein and Entrez from before)

In [18]:
aligner = Align.PairwiseAligner()
aligner.mode = "global"

human_protein = str(protein_records[0].seq)   # Homo sapiens was first

def identity(a, b):
    aln = aligner.align(a, b)[0]
    m = sum(x == y for x, y in zip(aln[0], aln[1]) if x != "-" and y != "-")
    return round(100 * m / len(aln[0]), 1)

for acc in ["NM_001270883.1", "NM_001270884.1"]:
    uid = Entrez.read(Entrez.esearch(db="nucleotide", term=acc))["IdList"][0]
    rec, cds, protein = get_cds_and_protein(uid)
    print(acc, "|", len(protein), "aa | identity to human HBB:", identity(human_protein, protein), "%")

NM_001270883.1 | 147 aa | identity to human HBB: 89.8 %
NM_001270884.1 | 147 aa | identity to human HBB: 89.8 %


compare the dog proteins to each other, and to human HBD


In [19]:
dog = {}
for acc in ["NM_001270883.1", "NM_001270884.1"]:
    uid = Entrez.read(Entrez.esearch(db="nucleotide", term=acc))["IdList"][0]
    rec, cds, protein = get_cds_and_protein(uid)
    dog[acc] = protein

# human HBD (delta) as a "decoy" to compare against
uid = Entrez.read(Entrez.esearch(db="nucleotide", term="NM_000519"))["IdList"][0]
rec, cds, human_hbd = get_cds_and_protein(uid)
print("Human HBD record:", rec.description[:70])

a, b = dog.values()
print("\nDog gene 1 vs dog gene 2 identity:", identity(a, b), "%")
print("Identical sequences?", a == b)

for acc, prot in dog.items():
    print(acc,
          "| vs human HBB:", identity(human_protein, prot), "%",
          "| vs human HBD:", identity(human_hbd, prot), "%")

Human HBD record: Homo sapiens hemoglobin subunit delta (HBD), mRNA

Dog gene 1 vs dog gene 2 identity: 100.0 %
Identical sequences? True
NM_001270883.1 | vs human HBB: 89.8 % | vs human HBD: 85.7 %
NM_001270884.1 | vs human HBB: 89.8 % | vs human HBD: 85.7 %


ompare the cow proteins to each other, and to human HBD

In [20]:
human = str(protein_records[0].seq)
cow = [r for r in protein_records if r.id == "Bos_taurus"][0].seq

aln = aligner.align(human, str(cow))[0]
print(aln[0])
print(aln[1])

MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
M--LTAEEKAAVTAFWGKVKVDEVGGEALGRLLVVYPWTQRFFESFGDLSTADAVMNNPKVKAHGKKVLDSFSNGMKHLDDLKGTFAALSELHCDKLHVDPENFKLLGNVLVVVLARNFGKEFTPVLQADFQKVVAGVANALAHRYH
